# Paise Trade - Deep Learning Training (Shen et al. 2020)

## Instructions
1. Upload `X_deep.npy` and `y_deep.npy` as a Dataset.
2. Creates `deep_model.h5` and `scaler.pkl` in Output.
3. **Enable GPU** in Accelerator settings.

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import joblib
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

# Patch np.object
try:
    np.object = object
except:
    pass

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
class DeepQuantModel:
    """
    Implements the 'Comprehensive Deep Learning System' (Shen et al. 2020).
    Core: Bidirectional LSTM to capture temporal dependencies (Trends/Cycles).
    """
    
    def __init__(self, input_shape):
        self.input_shape = input_shape
        self.model = self._build_model()
        
    def _build_model(self):
        model = Sequential()
        
        # Explicit Input Layer
        model.add(tf.keras.Input(shape=self.input_shape))
        
        # 1. Feature Extraction Layer (LSTM)
        # Bidirectional allows the model to see past and future contexts within the window
        model.add(Bidirectional(LSTM(128, return_sequences=True)))
        model.add(Dropout(0.3)) # Prevent overfitting
        model.add(BatchNormalization()) # Stabilize training
        
        # 2. Deep Temporal Layer
        model.add(LSTM(64, return_sequences=False))
        model.add(Dropout(0.3))
        
        # 3. Decision Layer (Dense)
        model.add(Dense(32, activation='relu'))
        
        # Output: Probability of "Profitable Signal"
        model.add(Dense(1, activation='sigmoid'))
        
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
        
        model.compile(
            optimizer=optimizer,
            loss='binary_crossentropy',
            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
        )
        
        return model
        
    def summary(self):
        return self.model.summary()
        
    def fit(self, X_train, y_train, X_val, y_val, epochs=50, batch_size=32, callbacks=None, verbose=1):
        # Use Early Stopping
        es_callback = tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', patience=10, restore_best_weights=True, mode='max'
        )
        
        final_callbacks = [es_callback]
        if callbacks:
            final_callbacks.extend(callbacks)
        
        history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=final_callbacks,
            verbose=verbose
        )
        return history
    
    def predict(self, X):
        return self.model.predict(X)
    
    def save(self, path):
        self.model.save(path)
        
    @staticmethod
    def load(path):
        return tf.keras.models.load_model(path)

In [ ]:
def train_deep_model():
    # 1. Load Data
    # Kaggle Input Dir - Update if dataset name is different
    input_dir = "/kaggle/input"
    
    # Find the .npy files recursively if needed, or hardcode typical path
    X_path = None
    y_path = None
    
    # Auto-find logic
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file == 'X_deep.npy':
                X_path = os.path.join(root, file)
            elif file == 'y_deep.npy':
                y_path = os.path.join(root, file)
    
    if not X_path or not y_path:
        print("❌ Data not found in /kaggle/input. Please upload X_deep.npy and y_deep.npy (Dataset).")
        return
        
    print(f"Loading X from {X_path} ...")
    X = np.load(X_path)
    y = np.load(y_path)
    
    print(f"Loaded Data: X={X.shape}, y={y.shape}")
    
    # 2. Scaling
    samples, timesteps, features = X.shape
    X_flat = X.reshape(-1, features)
    
    scaler = StandardScaler()
    X_flat_scaled = scaler.fit_transform(X_flat)
    X_scaled = X_flat_scaled.reshape(samples, timesteps, features)
    
    # Save Scaler to Output
    joblib.dump(scaler, 'scaler.pkl')
    print("Scaler saved to scaler.pkl")
    
    # 3. Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, shuffle=True, random_state=42)
    
    # 4. Feature Selection (RFE)
    print("Performing Recursive Feature Elimination (RFE) - Shen et al. 2020...")
    subset_size = min(10000, len(X_flat_scaled))
    X_subset = X_flat_scaled[:subset_size]
    
    X_last_step = X_train[:, -1, :] 
    y_rfe = y_train
    
    rfe_selector = RFE(estimator=LogisticRegression(solver='liblinear'), n_features_to_select=12)
    rfe_selector.fit(X_last_step, y_rfe)
    
    support = rfe_selector.support_
    print(f"RFE Selected {sum(support)} features: {np.where(support)[0]}")
    
    X_train = X_train[:, :, support]
    X_test = X_test[:, :, support]
    features = X_train.shape[2]
    
    # 5. Initialize Model
    model = DeepQuantModel(input_shape=(timesteps, features))
    model.summary()
    
    # 6. Train
    print("Starting Training on GPU...")
    history = model.fit(
        X_train, y_train,
        X_val=X_test, y_val=y_test,
        epochs=50, 
        batch_size=64,
        verbose=1
    )
    
    # 7. Evaluate
    loss, acc, auc = model.model.evaluate(X_test, y_test, verbose=0)
    print(f"Test Accuracy: {acc:.2%}, AUC: {auc:.4f}")
    
    # 8. Save Model
    model.save('deep_model.h5')
    print("Model saved to deep_model.h5")

if __name__ == "__main__":
    train_deep_model()